## Modelado.

In [1]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt
import json
from sklearn.model_selection import TimeSeriesSplit

In [2]:
train_viajeros = pd.read_csv("CSVs/train_viajeros.csv")
df_verano_viajeros = pd.read_csv("CSVs/verano_viajeros.csv")
df_viajeros_totales = pd.read_csv("CSVs/viajeros_totales.csv")

In [3]:
train_viajeros = (train_viajeros.sort_values(["Año", "Mes"]).reset_index(drop=True))

Definición de variables.

In [4]:
x_columnas = ["Provincias", "Año", "Mes", "log_lag_estacional", "log_lag_1", "log_lag_2", "log_lag_3", "log_lag_4", "media_lag_2", "media_lag_3", "tendencia_corta"] 
y_columnas = "log_ratio"

categoricas_viajeros = [x_columnas.index("Provincias")]

### Modelo base: Naive

In [5]:
datos_2024 = train_viajeros[train_viajeros["Año"] == 2024]
true_naive = datos_2024[y_columnas]

prediccion_naive = np.zeros(len(true_naive))

mae_naive = mean_absolute_error(true_naive, prediccion_naive)
rmse_naive = np.sqrt(mean_squared_error(true_naive, prediccion_naive))

print("--- MODELO NAIVE ---")
print("MAE:", mae_naive)
print("RMSE:", rmse_naive)

--- MODELO NAIVE ---
MAE: 0.17095569081802747
RMSE: 0.2377194249241485


## Algoritmos candidatos

### CatBoost

Selección de hiperparámetros: Grid Search con TimeSeriesSplit.

In [6]:

tscv = TimeSeriesSplit(n_splits=3)

depth_values = [4, 5, 6]
learning_rates = [0.02, 0.03, 0.04]
l2_values = [3, 4, 5, 6]

resultados_viajeros = []

for depth in depth_values:
    for learning in learning_rates:
        for l2 in l2_values:

            cv_rmses = []

            cv_maes = []

            for train_index, validacion_index in tscv.split(train_viajeros):

                x_train_fold = train_viajeros.iloc[train_index][x_columnas]
                y_train_fold = train_viajeros.iloc[train_index][y_columnas]

                x_validacion_fold = train_viajeros.iloc[validacion_index][x_columnas]
                y_validacion_fold = train_viajeros.iloc[validacion_index][y_columnas]

                modelo = CatBoostRegressor(
                    iterations = 700,
                    learning_rate = learning,
                    depth = depth,
                    l2_leaf_reg = l2,
                    loss_function = "RMSE",
                    random_seed = 42,
                    early_stopping_rounds = 50,
                    verbose=0
                )

                modelo.fit(x_train_fold, y_train_fold, cat_features=categoricas_viajeros, eval_set=(x_validacion_fold, y_validacion_fold))

                prediccion = modelo.predict(x_validacion_fold)

                cv_rmses.append(np.sqrt(mean_squared_error(y_validacion_fold, prediccion)))
                cv_maes.append(mean_absolute_error(y_validacion_fold, prediccion))

            avg_rmse = np.mean(cv_rmses)
            avg_mae = np.mean(cv_maes)
            
            print(f"Depth: {depth}, Learning_rate: {learning}, l2: {l2}")

            resultados_viajeros.append({"depth" : depth, "learning_rate" : learning, "l2" : l2, "MAE" : avg_mae, "RMSE" : avg_rmse})


Depth: 4, Learning_rate: 0.02, l2: 3
Depth: 4, Learning_rate: 0.02, l2: 4
Depth: 4, Learning_rate: 0.02, l2: 5
Depth: 4, Learning_rate: 0.02, l2: 6
Depth: 4, Learning_rate: 0.03, l2: 3
Depth: 4, Learning_rate: 0.03, l2: 4
Depth: 4, Learning_rate: 0.03, l2: 5
Depth: 4, Learning_rate: 0.03, l2: 6
Depth: 4, Learning_rate: 0.04, l2: 3
Depth: 4, Learning_rate: 0.04, l2: 4
Depth: 4, Learning_rate: 0.04, l2: 5
Depth: 4, Learning_rate: 0.04, l2: 6
Depth: 5, Learning_rate: 0.02, l2: 3
Depth: 5, Learning_rate: 0.02, l2: 4
Depth: 5, Learning_rate: 0.02, l2: 5
Depth: 5, Learning_rate: 0.02, l2: 6
Depth: 5, Learning_rate: 0.03, l2: 3
Depth: 5, Learning_rate: 0.03, l2: 4
Depth: 5, Learning_rate: 0.03, l2: 5
Depth: 5, Learning_rate: 0.03, l2: 6
Depth: 5, Learning_rate: 0.04, l2: 3
Depth: 5, Learning_rate: 0.04, l2: 4
Depth: 5, Learning_rate: 0.04, l2: 5
Depth: 5, Learning_rate: 0.04, l2: 6
Depth: 6, Learning_rate: 0.02, l2: 3
Depth: 6, Learning_rate: 0.02, l2: 4
Depth: 6, Learning_rate: 0.02, l2: 5
D

Selección de los mejores parámetros.

In [7]:
df_resultados_viajeros = pd.DataFrame(resultados_viajeros)
df_resultados_viajeros = df_resultados_viajeros.sort_values("RMSE")

mejores_parametros = df_resultados_viajeros.iloc[0]
mejor_rmse = mejores_parametros["RMSE"]

diferecnia = rmse_naive - mejores_parametros["RMSE"]

if mejores_parametros["RMSE"] < rmse_naive:
    print(f"El modelo mejora al Naive")
else:
    print("El modelo no mejora al Naive")

parametros = {"depth" : int(mejores_parametros["depth"]), "learning_rate" : float(mejores_parametros["learning_rate"]), "l2" : int(mejores_parametros["l2"])}

print("Mejores parámetros:")
print(mejores_parametros)

El modelo mejora al Naive
Mejores parámetros:
depth            6.000000
learning_rate    0.040000
l2               3.000000
MAE              0.169153
RMSE             0.237632
Name: 32, dtype: float64


In [8]:
archivo = "CSVs/parámetros_columnas.json"

datos = {
    "mejores_parametros" : parametros,
    "x_columnas" : x_columnas,
    "y_columnas" : y_columnas
}

with open (archivo, "w") as f:
    json.dump(datos, f, indent=4)
